<a href="https://colab.research.google.com/github/tusharsingh3199/ML-Projects/blob/main/BrainTumorSegmentation/BraTS_UNET_SwinUNETR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Library, Download Data, Patient Setup, EDA**

In [ ]:
import os
import zipfile
import random
import secrets
import nibabel as nib
from ipywidgets import interact, IntSlider
from matplotlib.widgets import Slider

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


In [ ]:
!kaggle datasets download --d aryashah2k/brain-tumor-segmentation-brats-2019

Dataset URL: https://www.kaggle.com/datasets/aryashah2k/brain-tumor-segmentation-brats-2019
License(s): CC0-1.0
100% 2.60G/2.60G [02:51<00:00, 16.3MB/s]



In [ ]:
Source_File = "/content/brain-tumor-segmentation-brats-2019.zip"
Data_Path = "/content/MICCAI_BraTS_2019_Data_Training"

if not os.path.exists(Data_Path):
    os.makedirs(Data_Path)
    with zipfile.ZipFile(Source_File, 'r') as zip_ref:
        zip_ref.extractall()

def patients_dir(Data_Path):
    patient_dir = []
    for grade in ["HGG", "LGG"]:
        grade_dir = os.path.join(Data_Path, grade)
        patients = sorted(os.listdir(grade_dir))
        patient_paths = [os.path.join(grade_dir, p) for p in patients]

        for path in patient_paths:
            patient_dir.append(path)

    return sorted(patient_dir)

patients_dir = patients_dir(Data_Path)
random.seed(40)
random.shuffle(patients_dir)


In [ ]:
def EDA(patients_dir):
    patient_dir = random.choice(patients_dir)
    img = []
    seg = []
    for modality in ["t1", "t1ce", "t2", "flair"]:
        v = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_{modality}.nii")).get_fdata().astype(np.float32)
        v = (v - v.min()) / (v.max() - v.min() + 1e-8)
        img.append(v)

    seg = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_seg.nii")).get_fdata().astype(np.int32)

    def view_slice(z):
        plt.figure(figsize=(12, 6))
        for i in range(4):
            plt.subplot(2, 4, i+1)
            plt.imshow(img[i][:, :, z], cmap="gray")
            plt.axis("off")

            plt.subplot(2, 4, 5+i)
            plt.imshow(img[i][:, :, z], cmap="gray")
            plt.axis("off")

            slice_seg = seg[:, :, z]
            masked_seg = np.ma.masked_where(slice_seg == 0, slice_seg)
            plt.imshow(masked_seg, cmap="jet", alpha=0.7, vmin=0, vmax=4)

        plt.show()

    interact(view_slice, z=IntSlider(min=0, max=154, step=1, value=75))

EDA(patients_dir)

# **Preparing the Dataset for 3D UNet, SWIN Transformer**

In [ ]:
train_dir = patients_dir[:int(len(patients_dir)*0.8)]
val_dir = patients_dir[int(len(patients_dir)*0.8):int(len(patients_dir)*0.9)]
test_dir = patients_dir[int(len(patients_dir)*0.9):]

def load_patient(patient_dir):
    patient_dir = patient_dir.numpy().decode()
    images = []
    for modality in ["t1", "t1ce", "t2", "flair"]:
        v = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_{modality}.nii")).get_fdata().astype(np.float32)
        v = (v - v.min()) / (v.max() - v.min() + 1e-8)
        images.append(v)

    image = np.stack(images, axis=-1).astype(np.float32)
    mask = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_seg.nii")).get_fdata().astype(np.int32)
    mask[mask == 4] = 3

    patch_size = (128, 128, 128)
    tumor = np.argwhere(mask > 0)

    if len(tumor) > 0:
        cx, cy, cz = tumor[np.random.randint(len(tumor))]
        x = np.clip(cx - patch_size[0] // 2, 0, image.shape[0] - patch_size[0])
        y = np.clip(cy - patch_size[1] // 2, 0, image.shape[1] - patch_size[1])
        z = np.clip(cz - patch_size[2] // 2, 0, image.shape[2] - patch_size[2])
    else:
        x = np.random.randint(0, image.shape[0] - patch_size[0] + 1)
        y = np.random.randint(0, image.shape[1] - patch_size[1] + 1)
        z = np.random.randint(0, image.shape[2] - patch_size[2] + 1)

    image = image[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2], :]
    mask = mask[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2]]

    return image, mask

def tf_load_patient(patient_dir):
    image, mask = tf.py_function(load_patient, [patient_dir], [tf.float32, tf.int32])
    image.set_shape((128, 128, 128, 4))
    mask.set_shape((128, 128, 128))
    return image, mask


train_dataset = (tf.data.Dataset.from_tensor_slices(train_dir).map(tf_load_patient, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(1).prefetch(tf.data.AUTOTUNE))

val_dataset = (tf.data.Dataset.from_tensor_slices(val_dir).map(tf_load_patient, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(1).prefetch(tf.data.AUTOTUNE))

test_dataset = (tf.data.Dataset.from_tensor_slices(test_dir).map(tf_load_patient)
                  .batch(1))

# **3D UNet Model Architecture**

In [ ]:
def conv_block(x, filters):
    x = tf.keras.layers.Conv3D(filters, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv3D(filters, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    return x

def decoder_block(x, skip, filters):
    x = tf.keras.layers.Conv3DTranspose(filters, kernel_size=2, strides=2, padding="same")(x)
    x = tf.keras.layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def Model_3UNet(input_shape=(128, 128, 128, 4), num_classes=4):
    inputs = tf.keras.Input(input_shape)

    s1 = conv_block(inputs, 16)
    p1 = tf.keras.layers.MaxPool3D(2)(s1)
    s2 = conv_block(p1, 32)
    p2 = tf.keras.layers.MaxPool3D(2)(s2)
    s3 = conv_block(p2, 64)
    p3 = tf.keras.layers.MaxPool3D(2)(s3)
    s4 = conv_block(p3, 128)
    p4 = tf.keras.layers.MaxPool3D(2)(s4)

    b = conv_block(p4, 256)

    d1 = decoder_block(b, s4, 128)
    d2 = decoder_block(d1, s3, 64)
    d3 = decoder_block(d2, s2, 32)
    d4 = decoder_block(d3, s1, 16)

    outputs = tf.keras.layers.Conv3D(num_classes, kernel_size=1, activation="softmax")(d4)
    return tf.keras.Model(inputs, outputs, name="3D_U-Net")

# **Swin UNETR Model**

In [ ]:
import numpy as np
import tensorflow as tf


def window_partition(x, window_size):
    B = tf.shape(x)[0]
    D, H, W, C = x.shape[1], x.shape[2], x.shape[3], x.shape[4]
    wd, wh, ww = window_size
    x = tf.reshape(x, [B, D // wd, wd, H // wh, wh, W // ww, ww, C])
    x = tf.transpose(x, [0, 1, 3, 5, 2, 4, 6, 7])
    return tf.reshape(x, [-1, wd, wh, ww, C])


def window_reverse(windows, window_size, D, H, W):
    wd, wh, ww = window_size
    C = windows.shape[-1]
    B = tf.shape(windows)[0] // ((D // wd) * (H // wh) * (W // ww))
    x = tf.reshape(windows, [B, D // wd, H // wh, W // ww, wd, wh, ww, C])
    x = tf.transpose(x, [0, 1, 4, 2, 5, 3, 6, 7])
    return tf.reshape(x, [B, D, H, W, C])


class WindowAttention3D(tf.keras.layers.Layer):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = tf.keras.layers.Dense(dim * 3, use_bias=qkv_bias)
        self.proj = tf.keras.layers.Dense(dim)

    def build(self, input_shape):
        wd, wh, ww = self.window_size
        self.num_tokens = wd * wh * ww
        self.relative_position_bias_table = self.add_weight(
            shape=((2 * wd - 1) * (2 * wh - 1) * (2 * ww - 1), self.num_heads),
            initializer="zeros", trainable=True, name="rel_pos_bias_table")

        coords = np.stack(np.meshgrid(np.arange(wd), np.arange(wh), np.arange(ww), indexing="ij"), axis=0)
        coords = coords.reshape(3, -1)
        rel = coords[:, :, None] - coords[:, None, :]
        rel = rel.transpose(1, 2, 0)
        rel[:, :, 0] += wd - 1
        rel[:, :, 1] += wh - 1
        rel[:, :, 2] += ww - 1
        rel[:, :, 0] *= (2 * wh - 1) * (2 * ww - 1)
        rel[:, :, 1] *= (2 * ww - 1)
        idx = rel.sum(-1).reshape(-1).astype(np.int32)
        self.relative_position_index = self.add_weight(
            shape=idx.shape, initializer=tf.keras.initializers.Constant(idx),
            trainable=False, dtype=tf.int32, name="rel_pos_index")
        super().build(input_shape)

    def call(self, x, mask=None):
        B_, N, C = tf.shape(x)[0], tf.shape(x)[1], self.dim
        qkv = tf.reshape(self.qkv(x), [B_, N, 3, self.num_heads, self.head_dim])
        qkv = tf.transpose(qkv, [2, 0, 3, 1, 4])
        q, k, v = qkv[0] * self.scale, qkv[1], qkv[2]
        attn = tf.matmul(q, k, transpose_b=True)

        bias = tf.gather(self.relative_position_bias_table, self.relative_position_index)
        bias = tf.transpose(tf.reshape(bias, [self.num_tokens, self.num_tokens, self.num_heads]), [2, 0, 1])
        attn = attn + bias[None, ...]

        if mask is not None:
            nW = tf.shape(mask)[0]
            attn = tf.reshape(attn, [B_ // nW, nW, self.num_heads, N, N]) + mask[None, :, None, :, :]
            attn = tf.reshape(attn, [-1, self.num_heads, N, N])

        attn = tf.nn.softmax(attn, axis=-1)
        out = tf.transpose(tf.matmul(attn, v), [0, 2, 1, 3])
        return self.proj(tf.reshape(out, [B_, N, C]))


class SwinTransformerBlock3D(tf.keras.layers.Layer):
    def __init__(self, dim, num_heads, window_size=(4, 4, 4), shift_size=(0, 0, 0), mlp_ratio=4.0, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.attn = WindowAttention3D(dim, window_size, num_heads)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(int(dim * mlp_ratio), activation="gelu"),
            tf.keras.layers.Dense(dim),
        ])

    def get_attn_mask(self, D, H, W):
        wd, wh, ww = self.window_size
        sd, sh, sw = self.shift_size
        if sd == 0 and sh == 0 and sw == 0:
            return None
        d_s = [(0, D - wd), (D - wd, D - sd), (D - sd, D)]
        h_s = [(0, H - wh), (H - wh, H - sh), (H - sh, H)]
        w_s = [(0, W - ww), (W - ww, W - sw), (W - sw, W)]
        mask = np.zeros((1, D, H, W, 1), dtype=np.float32)
        cnt = 0
        for d0, d1 in d_s:
            for h0, h1 in h_s:
                for w0, w1 in w_s:
                    if d1 > d0 and h1 > h0 and w1 > w0:
                        mask[:, d0:d1, h0:h1, w0:w1, :] = cnt
                    cnt += 1
        mw = tf.reshape(window_partition(tf.constant(mask), self.window_size), [-1, wd * wh * ww])
        diff = mw[:, None, :] - mw[:, :, None]
        return tf.where(diff != 0, tf.constant(-100.0), tf.constant(0.0))

    def call(self, x, D=None, H=None, W=None):
        B = tf.shape(x)[0]
        C = self.dim
        shortcut = x
        x = tf.reshape(self.norm1(x), [B, D, H, W, C])

        wd, wh, ww = self.window_size
        sd, sh, sw = self.shift_size
        pad_d, pad_h, pad_w = (wd - D % wd) % wd, (wh - H % wh) % wh, (ww - W % ww) % ww
        x = tf.pad(x, [[0, 0], [0, pad_d], [0, pad_h], [0, pad_w], [0, 0]])
        Dp, Hp, Wp = D + pad_d, H + pad_h, W + pad_w

        if sd or sh or sw:
            shifted = tf.roll(x, shift=[-sd, -sh, -sw], axis=[1, 2, 3])
            attn_mask = self.get_attn_mask(Dp, Hp, Wp)
        else:
            shifted, attn_mask = x, None

        windows = tf.reshape(window_partition(shifted, self.window_size), [-1, wd * wh * ww, C])
        attn_out = self.attn(windows, mask=attn_mask)
        attn_out = tf.reshape(attn_out, [-1, wd, wh, ww, C])
        shifted = window_reverse(attn_out, self.window_size, Dp, Hp, Wp)

        x = tf.roll(shifted, shift=[sd, sh, sw], axis=[1, 2, 3]) if (sd or sh or sw) else shifted
        if pad_d or pad_h or pad_w:
            x = x[:, :D, :H, :W, :]

        x = shortcut + tf.reshape(x, [B, D * H * W, C])
        return x + self.mlp(self.norm2(x))


class PatchMerging3D(tf.keras.layers.Layer):
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.reduction = tf.keras.layers.Dense(2 * dim, use_bias=False)
        self.norm = tf.keras.layers.LayerNormalization(epsilon=1e-5)

    def call(self, x, D=None, H=None, W=None):
        C = x.shape[-1]
        B = tf.shape(x)[0]
        x = tf.reshape(x, [B, D, H, W, C])
        parts = [x[:, i::2, j::2, k::2, :] for i in (0, 1) for j in (0, 1) for k in (0, 1)]
        x = tf.concat(parts, axis=-1)
        x = tf.reshape(x, [B, (D // 2) * (H // 2) * (W // 2), 8 * C])
        return self.reduction(self.norm(x))


def swin_stage(x, D, H, W, dim, depth, num_heads, window_size):
    shift = tuple(w // 2 for w in window_size)
    for i in range(depth):
        s = (0, 0, 0) if i % 2 == 0 else shift
        x = SwinTransformerBlock3D(dim, num_heads, window_size, s)(x, D=D, H=H, W=W)
    return x


def conv_block(x, filters, groups=8, dropout=0.0):
    x = tf.keras.layers.Conv3D(filters, 3, padding="same")(x)
    x = tf.keras.layers.GroupNormalization(groups=min(groups, filters))(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Conv3D(filters, 3, padding="same")(x)
    x = tf.keras.layers.GroupNormalization(groups=min(groups, filters))(x)
    x = tf.keras.layers.ReLU()(x)
    if dropout > 0:
        x = tf.keras.layers.SpatialDropout3D(dropout)(x)
    return x


def res_block(x, filters, groups=8):
    shortcut = x if x.shape[-1] == filters else tf.keras.layers.Conv3D(filters, 1, padding="same")(x)
    x = tf.keras.layers.Conv3D(filters, 3, padding="same")(x)
    x = tf.keras.layers.GroupNormalization(groups=min(groups, filters))(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Conv3D(filters, 3, padding="same")(x)
    x = tf.keras.layers.GroupNormalization(groups=min(groups, filters))(x)
    x = tf.keras.layers.Add()([x, shortcut])
    return tf.keras.layers.ReLU()(x)


def decoder_block(x, skip, filters):
    x = tf.keras.layers.Conv3DTranspose(filters, kernel_size=2, strides=2, padding="same")(x)
    x = tf.keras.layers.Concatenate()([x, skip])
    return res_block(x, filters)


class SwinEncoder(tf.keras.layers.Layer):
    def __init__(self, input_shape, patch_size, embed_dim, depths, num_heads, window_size, **kwargs):
        super().__init__(**kwargs)
        self.depths = depths
        self.num_heads = num_heads
        self.window_size = window_size
        self.embed_dim = embed_dim
        self.dims = [embed_dim * (2 ** i) for i in range(len(depths))]
        H0, W0, D0, _ = input_shape
        self.D0, self.H0, self.W0 = H0 // patch_size, W0 // patch_size, D0 // patch_size
        self.patch_embed = tf.keras.layers.Conv3D(embed_dim, kernel_size=patch_size, strides=patch_size, padding="same")
        self.merges = [PatchMerging3D(self.dims[i]) for i in range(len(depths) - 1)]

    def call(self, inputs):
        x = self.patch_embed(inputs)
        B = tf.shape(x)[0]
        D, H, W = self.D0, self.H0, self.W0
        x_seq = tf.reshape(x, [B, D * H * W, self.embed_dim])

        skips = []
        for i, (depth, heads) in enumerate(zip(self.depths, self.num_heads)):
            x_seq = swin_stage(x_seq, D, H, W, self.dims[i], depth, heads, self.window_size)
            skips.append(tf.reshape(x_seq, [B, D, H, W, self.dims[i]]))
            if i < len(self.depths) - 1:
                x_seq = self.merges[i](x_seq, D=D, H=H, W=W)
                D, H, W = D // 2, H // 2, W // 2
        return skips


def Model_SwinUNETR(
    input_shape=(128, 128, 128, 4),
    num_classes=4,
    patch_size=2,
    embed_dim=48,
    depths=(2, 2, 2, 2),
    num_heads=(3, 6, 12, 24),
    window_size=(4, 4, 4),
):
    inputs = tf.keras.Input(input_shape)
    dims = [embed_dim * (2 ** i) for i in range(len(depths))]

    enc0 = conv_block(inputs, embed_dim)

    skips = SwinEncoder(input_shape, patch_size, embed_dim, depths, num_heads, window_size)(inputs)
    bottleneck = skips[-1]
    proj_skips = [conv_block(s, dims[i]) for i, s in enumerate(skips[:-1])]

    d = conv_block(bottleneck, dims[-1], dropout=0.3)
    for i in reversed(range(len(depths) - 1)):
        d = decoder_block(d, proj_skips[i], dims[i])
    d = decoder_block(d, enc0, embed_dim)

    outputs = tf.keras.layers.Conv3D(num_classes, kernel_size=1, activation="softmax")(d)
    return tf.keras.Model(inputs, outputs, name="Swin_UNETR")

# **Model Training, Dice Score, Training Graphs**

In [ ]:
CLASS_WEIGHTS = tf.constant([0.05, 0.35, 0.2, 0.4], dtype=tf.float32)

def Dice_Class(y_true, y_pred, class_idx):
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=4)
    y_pred = tf.one_hot(tf.argmax(y_pred, axis=-1), depth=4)
    yt = y_true[..., class_idx]
    yp = y_pred[..., class_idx]

    smooth = 1e-6
    intersection = tf.reduce_sum(yt * yp, axis=[1,2,3])
    union = tf.reduce_sum(yt + yp, axis=[1,2,3])

    dice = (2. * intersection + smooth) / (union + smooth)
    return tf.reduce_mean(dice)

def Dice_NCR(y_true, y_pred):
    return Dice_Class(y_true, y_pred, 1)

def Dice_ED(y_true, y_pred):
    return Dice_Class(y_true, y_pred, 2)

def Dice_ET(y_true, y_pred):
    return Dice_Class(y_true, y_pred, 3)

def Dice(y_true, y_pred):
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=4)
    y_pred = tf.one_hot(tf.argmax(y_pred, axis=-1), depth=4)
    smooth = 1e-6
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])

    dice = (2. * intersection + smooth) / (union + smooth)
    return tf.reduce_mean(dice)

def Loss(y_true, y_pred):
    ce = tf.keras.losses.SparseCategoricalCrossentropy()(y_true, y_pred)
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=4)
    y_pred = tf.clip_by_value(y_pred, 1e-6, 1. - 1e-6)

    smooth = 1e-6
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])

    dice = (2. * intersection + smooth) / (union + smooth)
    weighted_dice = tf.reduce_sum(dice * CLASS_WEIGHTS) / tf.reduce_sum(CLASS_WEIGHTS)
    dice_loss = 1.0 - weighted_dice
    return dice_loss + ce

model_name = ["unet"]

if "unet" in model_name:
    model = Model_3UNet()
    callbacks = [
    tf.keras.callbacks.ModelCheckpoint("3D_UNet.keras", save_best_only=True, monitor="val_Dice", mode="max"),
    tf.keras.callbacks.EarlyStopping(monitor="val_Dice", mode="max", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_Loss", factor=0.5, patience=3, mode="min")
    ]

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=Loss, metrics=[Dice, Dice_NCR, Dice_ED, Dice_ET])
    history = model.fit(train_dataset, validation_data=val_dataset, epochs=10, callbacks=callbacks)

elif "swin_unetr" in model_name:
    model = Model_SwinUNETR()
    callbacks = [
    tf.keras.callbacks.ModelCheckpoint("Swin_UNETR.keras", save_best_only=True, monitor="val_dice", mode="max"),
    tf.keras.callbacks.EarlyStopping(monitor="val_dice", mode="max", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, mode="min")
    ]

    model.compile(optimizer=tf.keras.optimizers.AdamW(1e-4, weight_decay=1e-5, clipnorm=1.0), loss=Loss, metrics=[Dice, Dice_NCR, Dice_ED, Dice_ET])
    history = model.fit(train_dataset, validation_data=val_dataset, epochs=10, callbacks=callbacks)


In [ ]:
dice = history.history["dice"]
val_dice = history.history["val_dice"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(dice, label="Training Dice")
plt.plot(val_dice, label="Validation Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

model.evaluate(test_dataset)

# **Model Prediction and Results**

In [ ]:
def MRI_Results(patients_dir, model):
    patient_dir = random.choice(patients_dir)
    images = []
    patient_id = os.path.basename(patient_dir)

    for modality in ["t1", "t1ce", "t2", "flair"]:
        volume = nib.load(os.path.join(patient_dir, f"{patient_id}_{modality}.nii")).get_fdata().astype(np.float32)
        volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
        images.append(volume)

    image = np.stack(images, axis=-1)
    mask = nib.load(os.path.join(patient_dir, f"{patient_id}_seg.nii")).get_fdata().astype(np.int32)
    mask[mask == 4] = 3

    H, W, D, C = (240, 240, 155, 4)
    patch_size = (128, 128, 128)
    stride = (64, 64, 64)

    prediction = np.zeros((H, W, D, 4), dtype=np.float32)
    count_map = np.zeros((H, W, D, 1), dtype=np.float32)

    xs = list(range(0, H - patch_size[0] + 1, stride[0]))
    ys = list(range(0, W - patch_size[1] + 1, stride[1]))
    zs = list(range(0, D - patch_size[2] + 1, stride[2]))

    if xs[-1] != H - patch_size[0]:
        xs.append(H - patch_size[0])
    if ys[-1] != W - patch_size[1]:
        ys.append(W - patch_size[1])
    if zs[-1] != D - patch_size[2]:
        zs.append(D - patch_size[2])

    for x in xs:
        for y in ys:
            for z in zs:
                patch = image[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2], : ]
                patch = np.expand_dims(patch, axis=0)
                pred = model.predict(patch, verbose=0)[0]

                prediction[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2]] += pred
                count_map[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2]] += 1

    prediction /= count_map
    prediction = np.argmax(prediction, axis=-1)

    def view_slice(z):
        plt.figure(figsize=(8, 6))
        titles = ["T1","T1CE","T2","FLAIR"]

        for i in range(4):

            plt.subplot(3,4,i+1)
            plt.imshow(image[:,:,z,i], cmap="gray")
            plt.title(titles[i])
            plt.axis("off")

            plt.subplot(3,4,5+i)
            plt.imshow(image[:,:,z,i], cmap="gray")
            plt.imshow(np.ma.masked_where(mask[:,:,z]==0, mask[:,:,z]),
                       cmap="jet", alpha=0.6, vmin=0, vmax=3)
            plt.title("Ground Truth")
            plt.axis("off")

            plt.subplot(3,4,9+i)
            plt.imshow(image[:,:,z,i], cmap="gray")
            plt.imshow(np.ma.masked_where(prediction[:,:,z]==0, prediction[:,:,z]),
                       cmap="jet", alpha=0.6, vmin=0, vmax=3)
            plt.title("Prediction")
            plt.axis("off")

        plt.tight_layout()
        plt.show()

    interact(view_slice, z=IntSlider(min=0, max=154, value=64))

MRI_Results(patients_dir, model)